In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from statsforecast import StatsForecast
from statsforecast.utils import ConformalIntervals
from statsforecast.models import SeasonalExponentialSmoothing, ARIMA, ADIDA, GARCH, MSTL, MFLES, TBATS

from var import DATA_OUT, START_DATE

In [ ]:
df = pd.read_pickle(Path(DATA_OUT, 'df.pickle'))

## EDA

In [ ]:
df.columns.to_list()

In [ ]:
pd.plotting.autocorrelation_plot(df.loc['2024-01-03':'2024-01-07','s4_max'])

In [ ]:
# df['s4_mean'].diff(24 * 60)

pd.plotting.autocorrelation_plot(
    df['s4_mean'].diff(60 * 24).loc['2024-01-03':'2024-01-07']
)

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
adfuller(
    df.loc['2024-05-10':'2024-05-15','s4_mean'].fillna(0)
)

## Nixtla StatsForecast

In [ ]:
# df_plt = df.loc['2023', 's4_mean']

# fig, ax = plt.subplots(figsize=(16, 5))

# ax.plot(df_plt)
# ax.grid(True, linestyle='-', linewidth=0.4, alpha=0.5)
# ax.set_xlim(df_plt.index[0], df_plt.index[-1])
# ax.set_ylim(df_plt.min())

# plt.show()

In [ ]:
X_cols = ['h_tmk']
y_col = ['s4_mean']

In [ ]:
# df_train = df.loc['2024-02-01':'2024-02-15', y_col].copy().reset_index(names='ds').rename(columns={'s4_mean': 'y'})
# df_train["unique_id"] = "s4_forecast"

# df_test = df.loc['2024-02-18 19:30':'2024-02-20', y_col].copy().reset_index(names='ds').rename(columns={'s4_mean': 'y'})
# df_test["unique_id"] = "s4_forecast"

In [ ]:
df_train = df.loc[
    '2024-01-01':'2024-03-31', y_col
].copy().reset_index(names='ds').rename(
    columns={'s4_mean': 'y'}
).fillna(0)
df_train["unique_id"] = "s4_forecast"

df_test = df.loc[
    '2024-04-01':'2024-04-10', y_col
].copy().reset_index(names='ds').rename(
    columns={'s4_mean': 'y'}
)
df_test["unique_id"] = "s4_forecast"

In [ ]:
horizon = 15

intervals = ConformalIntervals(h=horizon, n_windows=2)

models = [
    # SeasonalExponentialSmoothing(season_length=10*24*60, alpha=0.1, prediction_intervals=intervals),
    # AutoARIMA(
    #     seasonal=True,
    #     season_length=24*60,
    #     stepwise=True,
    #     approximation=True,
    #     d=1,
    #     D=1,
    # )
    # ADIDA(),
    # ARIMA(
    #     order=(2,1,2),
    #     seasonal_order=(1,0,1),
    #     include_drift=True,
    #     prediction_intervals=intervals
    # ),
    MSTL(season_length=24*60)
]

sf = StatsForecast(
    models=models, 
    freq='1min',
    n_jobs=-1,
)

In [ ]:
sf.fit(df=df_train, prediction_intervals=intervals)

In [ ]:
df_pred = sf.predict(
    h=horizon,
    X_df=df_test.drop(columns=['y']).head(horizon),
    level=[90, 95],
)
df_pred['y'] = df_test['y'].head(horizon)

In [ ]:
model = 'MSTL'
cl = 95

plt.figure(figsize=(15, 5))
plt.plot(df_pred["ds"], df_pred[f"{model}"], label=f"{model}", c="tab:blue", ls='-', lw=0.8, marker='o')
plt.fill_between(df_pred["ds"], df_pred[f"{model}-lo-{cl}"], df_pred[f"{model}-hi-{cl}"], color="tab:blue", alpha=0.2)
plt.plot(df_pred["ds"], df_pred["y"], label="Real", c="k", ls="-", lw=0.8, marker="o")

plt.legend()
plt.grid()
plt.show()

## Idea: perché non usare il [modellino](https://colab.research.google.com/drive/1i9zpgOppIjNVN4K7xaFcFmlCiuFn5Pez) che ho fatto con MAPIE?

In [ ]:
from typing import cast

import numpy as np
import pandas as pd
from matplotlib import pylab as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from mapie._typing import NDArray
from mapie.metrics import regression_coverage_score, coverage_width_based, regression_mean_width_score
from mapie.regression import MapieTimeSeriesRegressor
from mapie.subsample import BlockBootstrap

In [ ]:
ALPHAS = [1 - 0.80, 1 - 0.90, 1 - 0.95]
GAP = 1

In [ ]:
df['s4_lag_10mins'] = df['s4_mean'].shift(10)

In [ ]:
X_cols = [
    'n_sat',
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_lag_10mins',
]

y_col = ['s4_mean']

X_train, X_test = df.loc['2024-01-01':'2024-03-31', X_cols].copy(), df.loc['2024-04-01':'2024-04-10', X_cols].copy()
y_train, y_test = df.loc['2024-01-01':'2024-03-31', y_col].copy().fillna(0), df.loc['2024-04-01':'2024-04-10', y_col].copy().fillna(0)

In [ ]:
# n_iter = 50
# n_splits = 5
# tscv = TimeSeriesSplit(n_splits=n_splits)
# random_state = 42
# rf_model = RandomForestRegressor(random_state=random_state)
# rf_params = {
#     "max_depth": [int(x) for x in np.linspace(2, 10, num=5)],
#     "n_estimators": [int(x) for x in np.linspace(10, 100, num=10)],
# }
# cv_obj = RandomizedSearchCV(
#     rf_model,
#     param_distributions=rf_params,
#     n_iter=n_iter,
#     cv=tscv,
#     scoring="neg_root_mean_squared_error",
#     random_state=random_state,
#     verbose=0,
#     n_jobs=-1,
# )
# cv_obj.fit(X_train, y_train.values)
# model = cv_obj.best_estimator_

In [ ]:
# model

In [ ]:
model = RandomForestRegressor(
    max_depth=4, n_estimators=30, random_state=42
)

In [ ]:
cv_mapietimeseries = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

In [ ]:
results_no_pfit = []
for alpha in ALPHAS:
    enbpi_no_pfit = MapieTimeSeriesRegressor(
        model,
        method='enbpi',
        cv=cv_mapietimeseries,
        agg_function='mean',
        n_jobs=-1,
    )

    enbpi_no_pfit.fit(X_train, y_train.values)

    y_pred_no_pfit, y_pis_no_pfit = enbpi_no_pfit.predict(
        X_test, alpha=alpha, ensemble=True,
    )

    results_no_pfit.append(
        {
            'y_pred': y_pred_no_pfit,
            'y_pis': y_pis_no_pfit,
            'coverage': regression_coverage_score(y_test, y_pis_no_pfit[:, 0, 0], y_pis_no_pfit[:, 1, 0]),
            'width': regression_mean_width_score(y_pis_no_pfit[:, 1, 0], y_pis_no_pfit[:, 0, 0]),
            'alpha': alpha,
        }
    )

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=2, label="Actual (test)", c="C1")

ax.plot(
    y_test.index,
    results_no_pfit[0]["y_pred"],
    lw=2,
    c="C2",
    label="Forecast"
)

for result_ in results_no_pfit:
    y_pis = cast(NDArray, result_["y_pis"])
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.2,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['width']:.2f})",
    )

ax.set_title('EnbPI, without partial_fit', fontweight="bold", size=18)
plt.xticks(rotation=45)

ax.legend(prop={'size': 10}, loc='upper right', frameon=False)

plt.show()